<p align="center">
  <h1 align="center">🍳 Cookbook 01: Deep Learning Auto-Compression (Real World)</h1>
  <p align="center">
    <strong>GradTracer vs. Random Pruning vs. Native Quantization</strong>
  </p>
</p>

---

In this recipe, we prove GradTracer's superiority by comparing it against standard baselines across **4 different models**. We have moved beyond simulation to measure **Real Disk Size (MB)** and **Real CPU Inference Latency**.

### Comparison Baselines (All measured after mini-finetune):
1. **Trained Baseline**: Model after the profiling steps (FP32).
2. **Random Pruning**: Pruning weights randomly (same sparsity as GradTracer, FP32).
3. **Naive INT8**: Real PyTorch Dynamic Quantization (INT8 applied uniformly to all inference layers).
4. **GradTracer Optimized**: Real Mixed-Precision Quantization (dynamically recommended bit-widths) applied on top of Optimal Saliency Pruning.

## 1. Setup Environment & Configuration

In [ ]:
# !pip install gradtracer torch transformers datasets scikit-learn numpy

In [ ]:
import os, tempfile, copy, json, time, numpy as np
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
from datasets import load_dataset
from torch.utils.data import DataLoader

from gradtracer import FlowTracker
from gradtracer.analyzers.compression import CompressionTracker
from gradtracer.analyzers.quantization import QuantizationAdvisor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cpu_device = torch.device('cpu')
print(f"Training on: {device}, Inference Benchmark on: {cpu_device}")

CONFIG = {
    "dataset_name": "rotten_tomatoes",     
    "max_length": 128,
    "batch_size": 32,
    "performance_floor": 0.95, 
    "profile_steps": 40
}

MODELS_TO_TEST = [
    "distilbert-base-uncased", 
    "bert-base-uncased",
    "albert-base-v2",
    "roberta-base"
]

## 2. Advanced Real Compression Utilities

In [ ]:
def evaluate_accuracy(model, loader, eval_device=device):
    model.eval()
    model = model.to(eval_device)
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            inputs = {'input_ids': batch['input_ids'].to(eval_device), 'attention_mask': batch['attention_mask'].to(eval_device)}
            outputs = model(**inputs)
            correct += (outputs.logits.argmax(dim=-1) == batch['label'].to(eval_device)).sum().item()
            total += batch['label'].size(0)
    return correct / total

def evaluate_latency_ms(model, loader, eval_device=cpu_device, num_batches=10):
    """Measure average real-world inference latency in milliseconds per batch."""
    model.eval()
    model = model.to(eval_device)
    latencies = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= num_batches: break
            inputs = {'input_ids': batch['input_ids'].to(eval_device), 'attention_mask': batch['attention_mask'].to(eval_device)}
            start = time.time()
            _ = model(**inputs)
            latencies.append((time.time() - start) * 1000)
    return sum(latencies) / len(latencies) if latencies else 0.0

def measure_real_size_mb(model):
    """Save model state explicitly to disk and measure actual file bytes."""
    with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as tmp:
        torch.save(model.state_dict(), tmp.name)
        size_mb = os.path.getsize(tmp.name) / (1024 * 1024)
    os.remove(tmp.name)
    return size_mb

def apply_naive_native_int8_quantization(model):
    """Apply Naive PyTorch Dynamic INT8 Quantization uniformly (baseline)."""
    model_cpu = copy.deepcopy(model).to(cpu_device)
    model_q = torch.quantization.quantize_dynamic(
        model_cpu, {nn.Linear}, dtype=torch.qint8
    )
    return model_q

def apply_random_pruning(model, amount):
    """Global unstructured random pruning."""
    if amount <= 0: return copy.deepcopy(model)
    model_p = copy.deepcopy(model)
    parameters_to_prune = []
    for module in model_p.modules():
        if isinstance(module, nn.Linear):
            parameters_to_prune.append((module, 'weight'))
    
    if parameters_to_prune:
        prune.global_unstructured(
            parameters_to_prune,
            pruning_method=prune.RandomUnstructured,
            amount=amount,
        )
        for module, name in parameters_to_prune:
            prune.remove(module, name)
    return model_p

## 3. The Grand Multi-Architecture Battle (Real Evaluation)
Comparing actual disk footprints and CPU inference times. 
Our **Mixed-Precision Quantization** uses GradTracer's `QuantizationAdvisor` to protect sensitive layers from quantizing too much, evaluating varying bit-widths configurations (FP32/16/8/4) rather than applying naive INT8 directly.

In [ ]:
summary = []

for m_name in MODELS_TO_TEST:
    print("=" * 80)
    print(f"🔄 Testing Architecture: {m_name}")
    print("=" * 80)
    
    # 1. Loading & Data Preparation
    try:
        tokenizer = AutoTokenizer.from_pretrained(m_name)
        model = AutoModelForSequenceClassification.from_pretrained(m_name, num_labels=2).to(device)
    except Exception as e:
        print(f"  [!] Skip {m_name}: {e}"); continue
        
    ds = load_dataset(CONFIG["dataset_name"])
    tokenized_ds = ds.map(lambda x: tokenizer(x["text"], padding="max_length", truncation=True, max_length=CONFIG["max_length"]), batched=True)
    tokenized_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    train_loader = DataLoader(tokenized_ds["train"], batch_size=CONFIG["batch_size"], shuffle=True)
    test_loader = DataLoader(tokenized_ds["validation" if "validation" in tokenized_ds else "test"], batch_size=64)
    
    # 2. Dynamic Profiling (Warm-up / Mini-Finetune)
    print(f"  -> Profiling dynamics for {CONFIG['profile_steps']} steps...")
    tracker = FlowTracker(model, track_gradients=True, track_interval=5)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
    model.train()
    for i, batch in enumerate(train_loader):
        optimizer.zero_grad()
        loss_obj = model(batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device), labels=batch['label'].to(device))
        loss = loss_obj.loss
        loss.backward(); tracker.step(loss.item()); optimizer.step()
        if i >= CONFIG['profile_steps']: break

    # ⭐️ Establish TRUE BASELINE and grab REAL benchmarks
    trained_baseline_model = copy.deepcopy(model)
    base_acc = evaluate_accuracy(trained_baseline_model, test_loader, device)
    orig_size_mb = measure_real_size_mb(trained_baseline_model)
    orig_lat_ms = evaluate_latency_ms(trained_baseline_model, test_loader, cpu_device)
    print(f"  ✅ Trained Baseline: Acc={base_acc*100:.2f}%, Real Disk Size={orig_size_mb:.1f}MB, CPU Lat={orig_lat_ms:.1f}ms")

    # 3. GradTracer Optimized Strategy
    comp_tracker = CompressionTracker(trained_baseline_model, eval_fn=lambda m: evaluate_accuracy(m, test_loader, device))
    res = comp_tracker.auto_compress(performance_floor=CONFIG["performance_floor"], search_strategy="binary", precision=0.05)
    gt_sparse = res.optimal_config.get('sparsity', 0.0)
    
    # ⭐️ Apply GradTracer's Native Mixed-Precision ON TOP of GradTracer's pruning
    q_advisor = QuantizationAdvisor(tracker)
    rec_plan = q_advisor.recommend_mixed_precision()
    gt_model_q = CompressionTracker.apply_mixed_precision_quantization(comp_tracker.model.to(cpu_device), rec_plan)
    
    gt_acc = evaluate_accuracy(gt_model_q, test_loader, cpu_device)
    gt_size_mb = measure_real_size_mb(gt_model_q)
    gt_lat_ms = evaluate_latency_ms(gt_model_q, test_loader, cpu_device)

    # 4. Fair Comparison vs Baselines (Same Starting State)
    print("  -> Evaluating Baselines (from Trained State)...")
    rand_model = apply_random_pruning(trained_baseline_model, gt_sparse)
    rand_acc = evaluate_accuracy(rand_model, test_loader, device)
    
    native_quant_model = apply_naive_native_int8_quantization(trained_baseline_model)
    quant_acc = evaluate_accuracy(native_quant_model, test_loader, cpu_device)
    quant_size_mb = measure_real_size_mb(native_quant_model)
    
    # 5. Collection
    summary.append({
        "model": m_name,
        "base_acc": base_acc, "orig_size": orig_size_mb,
        "gt_acc": gt_acc, "gt_size": gt_size_mb, "gt_lat": gt_lat_ms,
        "rand_acc": rand_acc,
        "quant_acc": quant_acc, "quant_size": quant_size_mb,
        "gt_sparse": gt_sparse
    })
    
    print(f"  🟠 GradTracer (Sparse + Mixed-Precision): Acc={gt_acc*100:.2f}%, Real Size={gt_size_mb:.1f}MB, CPU Lat={gt_lat_ms:.1f}ms")
    print(f"  🔴 Random Pruning (FP32)               : Acc={rand_acc*100:.2f}%")
    print(f"  🔴 Naive Uniform INT8                : Acc={quant_acc*100:.2f}%, Real Size={quant_size_mb:.1f}MB")
    print("\n")

## 4. Final Real Verification Dashboard

In [ ]:
print("=" * 115)
print(f" {'Model':<25} | {'Trained Base':<12} | {'GradTracer':<10} | {'Random':<8} | {'Naive INT8':<10} | {'Real Size Reduc'}")
print("-" * 115)
for s in summary:
    win = "🏆" if s['gt_acc'] >= s['quant_acc'] - 0.01 and s['gt_acc'] > s['rand_acc'] else ""
    reduction_pct = (1.0 - s['gt_size'] / s['orig_size']) * 100
    
    print(f" {s['model'][:25]:<25} | {s['base_acc']*100:>11.1f}% | {s['gt_acc']*100:>9.1f}% | {s['rand_acc']*100:>7.1f}% | {s['quant_acc']*100:>9.1f}% | {s['orig_size']:.1f}MB -> {s['gt_size']:.1f}MB ({reduction_pct:.1f}%) {win}")
print("=" * 115)